In [1]:
# here you can have your lil terminal#

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📦 IMPORTS & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("🚀 LSTM EPSS Full Sequence Evaluation")
print("=" * 50)


🚀 LSTM EPSS Full Sequence Evaluation


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 UTILITY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def get_device():
    """Get optimal device (CUDA if available, else CPU)"""
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        print("[INFO] GPU:", torch.cuda.get_device_name(0))
        return torch.device("cuda")
    print("[WARN] CUDA unavailable → CPU")
    return torch.device("cpu")


def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    """Cast DataFrame columns to optimal types and sort by CVE and date"""
    df = df.copy()
    df["date"]         = pd.to_datetime(df["date"], errors="raise")
    df["cve"]          = df["cve"].astype("category")
    df["epss"]         = df["epss"].astype("float32")
    df["age_epss_pub"] = df["age_epss_pub"].astype("int32")
    return df.sort_values(["cve", "date"]).reset_index(drop=True)


def transform_epss(
    arr: np.ndarray,
    mode: str = "inverted_log",
    eps: float = 1e-6
) -> np.ndarray:
    """
    Transform EPSS probabilities for better model training.
    
    Parameters:
    -----------
    arr : np.ndarray
        1D array of probabilities in [0,1]
    mode : str
        Transform type: "inverted_log" | "cloglog" | "logit"
    eps : float
        Clipping epsilon to avoid log(0) or division by zero
        
    Returns:
    --------
    np.ndarray
        Transformed values as float32
    """
    # Clip into (eps, 1-eps) to avoid numerical issues
    p = np.clip(arr, eps, 1.0 - eps)

    # Apply transform
    if mode == "inverted_log":
        # y = -log(p)
        out = -np.log(p)
    elif mode == "cloglog":
        # y = log(-log(1-p))
        out = np.log(-np.log(1.0 - p))
    elif mode == "logit":
        # y = log(p / (1-p))
        out = np.log(p / (1.0 - p))
    else:
        raise ValueError(f"Unknown transform: {mode!r}")

    return out.astype(np.float32)

# Test device availability
device = get_device()


[INFO] GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 DATA LOADING & PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════════════

# Configuration
FILE = "/notebooks/data/final_full_data.parquet"
TRANSFORM_MODE = "logit"  # Options: "inverted_log", "cloglog", "logit"

print(f"📂 Loading data from: {FILE}")
print(f"🔄 EPSS transform mode: {TRANSFORM_MODE}")

# Load and preprocess data
BIG = cast_types(pd.read_parquet(FILE, engine="pyarrow"))

# Apply EPSS transformation
BIG["epss"] = transform_epss(BIG["epss"].values, mode=TRANSFORM_MODE, eps=1e-6)

print(f"✅ Loaded full table: {BIG.shape} (epss transformed via {TRANSFORM_MODE})")
print(f"📈 Data range: {BIG['date'].min().date()} to {BIG['date'].max().date()}")
print(f"🎯 Unique CVEs: {BIG['cve'].nunique():,}")
print(f"📊 EPSS stats after transform: min={BIG['epss'].min():.3f}, max={BIG['epss'].max():.3f}, mean={BIG['epss'].mean():.3f}")

# Display sample
print("\n📋 Sample data:")
print(BIG.head())


📂 Loading data from: /notebooks/data/final_full_data.parquet
🔄 EPSS transform mode: logit


FileNotFoundError: [Errno 2] No such file or directory: '/notebooks/data/final_full_data.parquet'

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📅 TEMPORAL SPLITS (LEAKAGE-FREE)
# ═══════════════════════════════════════════════════════════════════════════════

# Create calendar-based splits to prevent temporal leakage
days = np.sort(BIG["date"].unique())
n_days = len(days)

# Split points: 64% train, 16% val, 20% test
TEST_CUT = days[int(0.80 * n_days)]
VAL_CUT = days[int(0.64 * n_days)]

VAL_CUT = pd.to_datetime(VAL_CUT)
TEST_CUT = pd.to_datetime(TEST_CUT)

print(f"📊 Temporal Split Strategy:")
print(f"   🟢 TRAIN: < {VAL_CUT.date()}")
print(f"   🟡 VAL:   {VAL_CUT.date()} to {TEST_CUT.date()}")
print(f"   🔴 TEST:  >= {TEST_CUT.date()}")
print(f"   📈 Total days: {n_days:,}")

# Create loss flags for each split
def flag(df, cond):
    """Create binary flag array for given condition"""
    out = np.zeros(len(df), np.float32)
    out[cond] = 1.0
    return out

train_flag = flag(BIG, BIG["date"] < VAL_CUT)
val_flag = flag(BIG, (BIG["date"] >= VAL_CUT) & (BIG["date"] < TEST_CUT))
test_flag = flag(BIG, BIG["date"] >= TEST_CUT)

# Create datasets with loss flags
df_tr = BIG.copy()
df_tr["use_for_loss"] = train_flag

df_val = BIG.copy()
df_val["use_for_loss"] = val_flag

df_te = BIG.copy()
df_te["use_for_loss"] = test_flag

# Display split statistics
for name, df in zip(("TRAIN", "VAL", "TEST"), (df_tr, df_val, df_te)):
    loss_rows = (df["use_for_loss"] == 1).sum()
    unique_cves = df["cve"].nunique()
    print(f"   {name:<5}: {len(df):,} total rows | {loss_rows:,} loss rows | {unique_cves:,} CVEs")

print(f"\n✅ Temporal splits created with no future leakage")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔧 FEATURE STANDARDIZATION
# ═══════════════════════════════════════════════════════════════════════════════

# Standardize age_epss_pub feature using training data only
print("🔧 Standardizing features...")

scaler = StandardScaler().fit(df_tr[["age_epss_pub"]])

# Apply standardization to all splits
for name, df in zip(("train", "val", "test"), (df_tr, df_val, df_te)):
    original_mean = df["age_epss_pub"].mean()
    df["age_epss_pub"] = scaler.transform(df[["age_epss_pub"]])
    new_mean = df["age_epss_pub"].mean()
    new_std = df["age_epss_pub"].std()
    print(f"   {name:<5}: age_epss_pub {original_mean:.2f} → μ={new_mean:.3f}, σ={new_std:.3f}")

print("✅ Feature standardization complete")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🗂️ DATASET CLASS (FULL SEQUENCE WITH MASKING)
# ═══════════════════════════════════════════════════════════════════════════════

class CVEFullDataset(Dataset):
    """
    Dataset for full-sequence CVE EPSS prediction with temporal masking.
    
    Returns tuple (X, Y, m_t, m_h, m_eval) where:
    - X      : [L_max, 2] - Input features (epss, age_epss_pub)
    - Y      : [L_max, H] - Target values (future epss scores)
    - m_t    : [L_max]    - Temporal mask (real rows vs padding)
    - m_h    : [L_max, H] - Horizon mask (future availability)
    - m_eval : [L_max]    - Evaluation mask (1 if row counts toward loss)
    """
    
    def __init__(self, df, L_max: int, horizon: int = 10):
        """
        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with CVE data including 'use_for_loss' column
        L_max : int
            Maximum sequence length (for padding)
        horizon : int
            Number of future time steps to predict
        """
        self.X, self.Y, self.m_t, self.m_h, self.m_eval = [], [], [], [], []
        
        print(f"🔄 Processing {df['cve'].nunique():,} CVEs for dataset creation...")
        
        for cve_idx, (cve_id, g) in enumerate(df.groupby("cve", observed=True)):
            if cve_idx % 1000 == 0 and cve_idx > 0:
                print(f"   Processed {cve_idx:,} CVEs...")
            
            # Extract features and evaluation mask
            vals = g[["epss", "age_epss_pub"]].to_numpy("float32")
            evalm = g["use_for_loss"].to_numpy("float32")
            
            T = len(vals)
            pad = L_max - T
            
            # Pad sequences to L_max length
            self.X.append(torch.from_numpy(np.pad(vals, ((0, pad), (0, 0)), "constant")))
            self.m_t.append(torch.from_numpy(np.r_[np.ones(T), np.zeros(pad)].astype("float32")))
            self.m_eval.append(torch.from_numpy(np.r_[evalm, np.zeros(pad)].astype("float32")))
            
            # Create future targets and horizon mask
            Y = np.zeros((L_max, horizon), np.float32)
            mh = np.zeros_like(Y)
            
            for t in range(T):
                k = min(horizon, T - t - 1)  # Available future steps
                if k > 0:
                    Y[t, :k] = vals[t+1:t+1+k, 0]  # Future EPSS values
                    mh[t, :k] = 1  # Mark as available
            
            self.Y.append(torch.from_numpy(Y))
            self.m_h.append(torch.from_numpy(mh))
        
        print(f"✅ Dataset created: {len(self)} CVEs, L_max={L_max}, horizon={horizon}")
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, i):
        return (self.X[i], self.Y[i], self.m_t[i], self.m_h[i], self.m_eval[i])


def collate(batch):
    """Custom collate function to stack tensors"""
    return tuple(torch.stack(x, 0) for x in zip(*batch))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🧠 LSTM MODEL ARCHITECTURE
# ═══════════════════════════════════════════════════════════════════════════════

class Seq2SeqLSTM(nn.Module):
    """
    Sequence-to-sequence LSTM for multi-horizon EPSS prediction.
    
    Architecture:
    - Multi-layer LSTM encoder
    - Linear projection head for multi-horizon output
    - Dropout for regularization
    """
    
    def __init__(self, input_dim=2, hidden_dim=128, layers=2, 
                 horizon=10, dropout=0.3):
        """
        Parameters:
        -----------
        input_dim : int
            Number of input features (epss, age_epss_pub)
        hidden_dim : int
            LSTM hidden dimension
        layers : int
            Number of LSTM layers
        horizon : int
            Number of future time steps to predict
        dropout : float
            Dropout rate for regularization
        """
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.layers = layers
        self.horizon = horizon
        
        # LSTM encoder
        self.lstm = nn.LSTM(
            input_dim, 
            hidden_dim, 
            layers,
            batch_first=True, 
            dropout=dropout if layers > 1 else 0
        )
        
        # Multi-horizon prediction head
        self.head = nn.Linear(hidden_dim, horizon)
        
        print(f"🧠 LSTM Model Architecture:")
        print(f"   📥 Input dim: {input_dim}")
        print(f"   🔄 Hidden dim: {hidden_dim}")
        print(f"   📚 Layers: {layers}")
        print(f"   🎯 Horizon: {horizon}")
        print(f"   🎲 Dropout: {dropout}")
        
    def forward(self, x):
        """
        Forward pass
        
        Parameters:
        -----------
        x : torch.Tensor
            Input tensor of shape [batch, seq_len, input_dim]
            
        Returns:
        --------
        torch.Tensor
            Predictions of shape [batch, seq_len, horizon]
        """
        # LSTM encoding
        h, _ = self.lstm(x)  # [batch, seq_len, hidden_dim]
        
        # Multi-horizon prediction
        out = self.head(h)   # [batch, seq_len, horizon]
        
        return out


def masked_mse(pred, true, m_t, m_h, m_eval):
    """
    Compute masked MSE loss that honors evaluation and horizon masks.
    
    Parameters:
    -----------
    pred : torch.Tensor
        Predictions [batch, seq_len, horizon]
    true : torch.Tensor
        True values [batch, seq_len, horizon]
    m_t : torch.Tensor
        Temporal mask [batch, seq_len] - 1 for real data, 0 for padding
    m_h : torch.Tensor
        Horizon mask [batch, seq_len, horizon] - 1 for available future, 0 otherwise
    m_eval : torch.Tensor
        Evaluation mask [batch, seq_len] - 1 for rows to include in loss
        
    Returns:
    --------
    torch.Tensor
        Masked MSE loss (scalar)
    """
    # Combine temporal and evaluation masks
    mask = m_t * m_eval  # [batch, seq_len]
    
    # Compute squared error
    err = (pred - true) ** 2  # [batch, seq_len, horizon]
    
    # Apply masks
    err = err * mask.unsqueeze(-1) * m_h  # [batch, seq_len, horizon]
    
    # Return normalized loss
    return err.sum() / m_h.sum()

print("✅ Model architecture and loss function defined")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ⚙️ TRAINING CONFIGURATION & DATASET PREPARATION
# ═══════════════════════════════════════════════════════════════════════════════

# Set random seed for reproducibility
torch.manual_seed(0)

# Training hyperparameters
HORIZON = 10      # Number of future time steps to predict
BATCH_SIZE = 64   # Batch size for training
EPOCHS = 12       # Number of training epochs
LEARNING_RATE = 1e-3  # Learning rate

print(f"⚙️ Training Configuration:")
print(f"   🎯 Horizon: {HORIZON} time steps")
print(f"   📦 Batch size: {BATCH_SIZE}")
print(f"   🔄 Epochs: {EPOCHS}")
print(f"   📈 Learning rate: {LEARNING_RATE}")
print(f"   🎲 Random seed: 0")

# Calculate maximum sequence length across all CVEs
L_max = BIG.groupby("cve", observed=True).size().max()
print(f"   📏 Max sequence length: {L_max}")

# Create datasets
print(f"\n🗂️ Creating datasets...")
tr_ds = CVEFullDataset(df_tr, L_max, HORIZON)
va_ds = CVEFullDataset(df_val, L_max, HORIZON)
te_ds = CVEFullDataset(df_te, L_max, HORIZON)

# Create data loaders
print(f"\n🔄 Creating data loaders...")
tr_ld = DataLoader(
    tr_ds, 
    BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate,
    num_workers=0,  # Set to 0 for Windows compatibility
    pin_memory=True
)

va_ld = DataLoader(
    va_ds, 
    BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate,
    num_workers=0,
    pin_memory=True
)

te_ld = DataLoader(
    te_ds, 
    BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate,
    num_workers=0,
    pin_memory=True
)

print(f"✅ Data loaders created:")
print(f"   🟢 Train: {len(tr_ld)} batches")
print(f"   🟡 Val:   {len(va_ld)} batches")
print(f"   🔴 Test:  {len(te_ld)} batches")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🏗️ MODEL INITIALIZATION & OPTIMIZATION SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Initialize model
model = Seq2SeqLSTM(horizon=HORIZON).to(device)

# Enable torch.compile for CUDA if available (PyTorch 2.0+)
if hasattr(torch, "compile") and device.type == "cuda":
    print("🚀 Enabling torch.compile for optimized training...")
    model = torch.compile(model)
else:
    print("ℹ️ torch.compile not available or not using CUDA")

# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Display model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"   🔢 Total parameters: {total_params:,}")
print(f"   🎯 Trainable parameters: {trainable_params:,}")
print(f"   💾 Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB")
print(f"   🖥️ Device: {device}")

print(f"\n✅ Model and optimizer ready for training!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════

print("🚀 Starting training...")
print("=" * 60)

# Training history
train_losses = []
val_losses = []

for epoch in range(1, EPOCHS + 1):
    # ─────────────────────────────────────────────────────────────────────────
    # TRAINING PHASE
    # ─────────────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    
    for batch_idx, (X, Y, mt, mh, me) in enumerate(tqdm(tr_ld, desc=f"🟢 Train {epoch:02d}/{EPOCHS}")):
        # Move to device
        X, Y, mt, mh, me = (z.to(device, non_blocking=True) for z in (X, Y, mt, mh, me))
        
        # Forward pass
        optimizer.zero_grad()
        predictions = model(X)
        loss = masked_mse(predictions, Y, mt, mh, me)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping for stability
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Optimizer step
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(tr_ld)
    train_losses.append(train_loss)
    
    # ─────────────────────────────────────────────────────────────────────────
    # VALIDATION PHASE
    # ─────────────────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for X, Y, mt, mh, me in tqdm(va_ld, desc=f"🟡 Val   {epoch:02d}/{EPOCHS}"):
            # Move to device
            X, Y, mt, mh, me = (z.to(device, non_blocking=True) for z in (X, Y, mt, mh, me))
            
            # Forward pass
            predictions = model(X)
            loss = masked_mse(predictions, Y, mt, mh, me)
            
            val_loss += loss.item()
    
    val_loss /= len(va_ld)
    val_losses.append(val_loss)
    
    # ─────────────────────────────────────────────────────────────────────────
    # EPOCH SUMMARY
    # ─────────────────────────────────────────────────────────────────────────
    print(f"📊 Epoch {epoch:02d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    
    # Early stopping check (simple version)
    if epoch > 3 and val_loss > max(val_losses[-3:-1]):
        print(f"⚠️ Validation loss increasing - consider early stopping")

print("\n✅ Training completed!")
print(f"📈 Final train loss: {train_losses[-1]:.4f}")
print(f"📉 Final val loss: {val_losses[-1]:.4f}")
print(f"🏆 Best val loss: {min(val_losses):.4f} (epoch {val_losses.index(min(val_losses))+1})")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 TEST EVALUATION & RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

print("🔍 Evaluating on test set...")
print("=" * 50)

# Set model to evaluation mode
model.eval()

# Collect predictions and targets
pred_list, true_list, mask_h_list, eval_list = [], [], [], []

with torch.no_grad():
    for batch_idx, (X, Y, mt, mh, me) in enumerate(tqdm(te_ld, desc="🔴 Test Evaluation")):
        # Move to device and get predictions
        X = X.to(device, non_blocking=True)
        predictions = model(X).cpu()  # Move back to CPU for storage
        
        # Store results
        pred_list.append(predictions)
        true_list.append(Y)
        mask_h_list.append(mh)
        eval_list.append(me)

# Concatenate all batches
P = torch.cat(pred_list)   # Predictions
T = torch.cat(true_list)   # True values
MH = torch.cat(mask_h_list)  # Horizon mask
ME = torch.cat(eval_list)    # Evaluation mask

print(f"✅ Test evaluation complete!")
print(f"📊 Collected predictions for {P.shape[0]:,} CVE sequences")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📈 COMPUTE FINAL METRICS & MEMORY ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

print("🧮 Computing final test metrics...")

# Debug tensor sizes for memory analysis
print(f"\n🔍 Memory Analysis:")
print(f"   P (predictions): {P.shape} | {P.numel() * 4 / 1024 / 1024:.1f} MB")
print(f"   T (true values): {T.shape} | {T.numel() * 4 / 1024 / 1024:.1f} MB") 
print(f"   MH (horizon mask): {MH.shape} | {MH.numel() * 4 / 1024 / 1024:.1f} MB")
print(f"   ME (eval mask): {ME.shape} | {ME.numel() * 4 / 1024 / 1024:.1f} MB")

# Calculate combined mask for test evaluation
print(f"\n🎯 Computing masked metrics...")
m_comb = MH * ME.unsqueeze(-1)  # Combine horizon and evaluation masks

# Compute errors
squared_err = (P - T) ** 2
abs_err = (P - T).abs()

# Compute masked metrics
mse = (squared_err * m_comb).sum() / m_comb.sum()
mae = (abs_err * m_comb).sum() / m_comb.sum()

print(f"\n🏆 FINAL TEST RESULTS:")
print(f"   📊 MSE (Mean Squared Error): {mse:.6f}")
print(f"   📊 MAE (Mean Absolute Error): {mae:.6f}")
print(f"   📊 RMSE (Root Mean Squared Error): {mse.sqrt():.6f}")

# Additional statistics
valid_predictions = m_comb.sum().item()
total_possible = MH.numel()
coverage = valid_predictions / total_possible * 100

print(f"\n📈 Coverage Statistics:")
print(f"   ✅ Valid predictions: {valid_predictions:,.0f}")
print(f"   📊 Total possible: {total_possible:,}")
print(f"   📈 Coverage: {coverage:.2f}%")

print(f"\n✅ Evaluation complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 💾 SAVE RESULTS & MODEL ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════

print("💾 Saving results and model artifacts...")

# Get CVE IDs in the same order as the dataset constructor
ordered_cves = list(df_te.groupby("cve", observed=True).groups.keys())

# Save comprehensive results
results_file = "predictions_fullseq.npz"
np.savez(
    results_file,
    cves=np.array(ordered_cves, dtype=object),  # CVE ID mapping
    pred=P.numpy(),                             # Predictions
    true=T.numpy(),                             # True values  
    mask_h=MH.numpy(),                          # Horizon mask
    eval_mask=ME.numpy(),                       # Evaluation mask
    train_losses=np.array(train_losses),        # Training history
    val_losses=np.array(val_losses),            # Validation history
    final_mse=mse.item(),                       # Final MSE
    final_mae=mae.item(),                       # Final MAE
    config={
        'transform_mode': TRANSFORM_MODE,
        'horizon': HORIZON,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'l_max': L_max
    }
)

print(f"✅ Results saved to: {results_file}")
print(f"📊 File contains:")
print(f"   🎯 Predictions for {len(ordered_cves):,} CVEs")
print(f"   📈 Training history ({len(train_losses)} epochs)")
print(f"   🔧 Model configuration")
print(f"   📊 Final metrics (MSE: {mse:.6f}, MAE: {mae:.6f})")

# Optional: Save model state dict
model_file = "lstm_model_fullseq.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'input_dim': 2,
        'hidden_dim': 128,
        'layers': 2,
        'horizon': HORIZON,
        'dropout': 0.3
    },
    'final_mse': mse.item(),
    'final_mae': mae.item(),
    'epoch': EPOCHS
}, model_file)

print(f"✅ Model saved to: {model_file}")

print(f"\n🎉 LSTM EPSS Full Sequence Evaluation Complete!")
print(f"🏆 Final Performance: MSE={mse:.6f}, MAE={mae:.6f}")
print(f"📁 Results: {results_file}")
print(f"🧠 Model: {model_file}")
